# Table Extraction and Analysis

This notebook demonstrates how to work with tables from parsed documents,
including data cleaning, transformation, and SQL analysis.

In [ ]:
import pandas as pd
from analysis_helpers import (
    MinioExplorer,
    SQLContext,
    display_table_preview,
)

## 1. Load Tables from a Document

In [ ]:
explorer = MinioExplorer()

# List available data
agencies = explorer.list_agencies("parsed-zone")
print("Agencies:", agencies)

if agencies:
    assets = explorer.list_assets(agencies[0], "parsed-zone")
    print(f"\nAssets for {agencies[0]}:")
    for a in assets:
        print(f"  - {a.asset} ({len(a.versions)} versions)")

In [ ]:
# Load a document (replace with your actual data)
if agencies and assets:
    doc = explorer.load_document(agencies[0], assets[0].asset)
    
    # List all tables
    table_info = explorer.list_tables(doc)
    print(f"Found {len(table_info)} tables:")
    for t in table_info:
        print(f"  [{t['index']}] {t['title']}: {t['rows']} rows x {t['columns']} cols")

## 2. Explore Table Structure

In [ ]:
# Load the first table
if 'doc' in dir() and table_info:
    df = explorer.load_table(doc, table_index=0)
    
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)

In [ ]:
# Preview the data
if 'df' in dir():
    display_table_preview(df, title="Raw Table Data")

## 3. Data Cleaning

In [ ]:
# Common cleaning operations
if 'df' in dir():
    df_clean = df.copy()
    
    # Clean column names (lowercase, replace spaces)
    df_clean.columns = [
        col.lower().replace(' ', '_').replace('-', '_') 
        for col in df_clean.columns
    ]
    
    print("Cleaned columns:")
    print(df_clean.columns.tolist())

In [ ]:
# Check for missing values
if 'df_clean' in dir():
    missing = df_clean.isnull().sum()
    if missing.any():
        print("Missing values:")
        print(missing[missing > 0])
    else:
        print("No missing values")

In [ ]:
# Try to convert numeric columns
if 'df_clean' in dir():
    for col in df_clean.columns:
        # Try numeric conversion
        try:
            # Remove commas and convert
            if df_clean[col].dtype == 'object':
                cleaned = df_clean[col].astype(str).str.replace(',', '', regex=False)
                df_clean[col] = pd.to_numeric(cleaned, errors='ignore')
        except Exception:
            pass
    
    print("Updated data types:")
    print(df_clean.dtypes)

## 4. SQL Analysis

In [ ]:
# Create SQL context
if 'df_clean' in dir():
    ctx = SQLContext()
    ctx.register("data", df_clean)
    
    # Describe the table
    ctx.describe("data")

In [ ]:
# Basic statistics (modify query for your columns)
if 'ctx' in dir():
    # Get column names to build a dynamic query
    cols = df_clean.columns.tolist()
    print("Available columns:", cols)
    
    # Count rows
    result = ctx.query("SELECT COUNT(*) as row_count FROM data")
    print(f"\nTotal rows: {result['row_count'].iloc[0]}")

In [ ]:
# Example: Find unique values in first column (modify for your data)
if 'ctx' in dir() and cols:
    first_col = cols[0]
    result = ctx.query(f'''
        SELECT "{first_col}", COUNT(*) as count
        FROM data
        GROUP BY "{first_col}"
        ORDER BY count DESC
        LIMIT 10
    ''')
    display_table_preview(result, title=f"Top values in {first_col}")

## 5. Load Multiple Tables

In [ ]:
# Load all tables from the document
if 'doc' in dir():
    all_tables = {}
    for t in table_info:
        df_t = explorer.load_table(doc, t['index'])
        # Clean column names
        df_t.columns = [
            col.lower().replace(' ', '_').replace('-', '_') 
            for col in df_t.columns
        ]
        all_tables[f"table_{t['index']}"] = df_t
        print(f"Loaded table_{t['index']}: {df_t.shape}")

In [ ]:
# Register all tables for SQL
if 'all_tables' in dir():
    multi_ctx = SQLContext()
    for name, df_t in all_tables.items():
        multi_ctx.register(name, df_t)
    
    print("Registered tables:", multi_ctx.tables())

In [ ]:
# Example: Cross-table query (modify for your data)
if 'multi_ctx' in dir() and len(all_tables) >= 2:
    # Show first few rows from each table
    for name in multi_ctx.tables()[:2]:
        result = multi_ctx.query(f"SELECT * FROM {name} LIMIT 3")
        display_table_preview(result, title=name)

## 6. Export Results

In [ ]:
# Export cleaned data to CSV
if 'df_clean' in dir():
    output_path = "/home/jovyan/work/cleaned_data.csv"
    df_clean.to_csv(output_path, index=False)
    print(f"Exported to {output_path}")

In [ ]:
# Export to Parquet (more efficient for large datasets)
if 'df_clean' in dir():
    parquet_path = "/home/jovyan/work/cleaned_data.parquet"
    df_clean.to_parquet(parquet_path, index=False)
    print(f"Exported to {parquet_path}")

## Next Steps

- See `03_metadata_curation.ipynb` to curate metadata for your tables
- Use the cleaned DataFrames for visualization or further analysis